# Mesclar adapter + quantizar pra GGUF (Q4) — pronto pro Ollama

Caminho único, sem alternativas: pega o adapter treinado (`Meu Drive/outputs-logos-1-v2/adapter`),
mescla nos pesos do modelo base em precisão cheia (bfloat16 — mesclar sobre pesos já
quantizados em 4-bit seria impreciso), converte pra GGUF e quantiza pra **Q4_K_M** (mesmo
nível do `granite4.1:8b` que você já tem no Ollama). No final você baixa só o `.gguf`
quantizado (~5GB) — os artefatos intermediários (modelo mesclado bf16 ~16GB, GGUF f16 ~16GB)
ficam só no disco temporário do Colab, nunca baixados.

**Runtime: GPU (L4, ~16-20GB livres) pra célula de merge. Custo: alto só nessa célula**
(carrega o modelo base inteiro em bfloat16). As células de conversão/quantização depois são
CPU-only e baratas.

## 0. Ambiente

In [ ]:
import os


def _running_on_kaggle() -> bool:
    return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _running_on_colab() -> bool:
    if _running_on_kaggle():
        return False
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


ON_COLAB = _running_on_colab()
print(f"Colab: {ON_COLAB}")
assert ON_COLAB, "este notebook espera rodar no Colab (precisa da GPU pra célula de merge).

## 1. Instalar dependências e reiniciar o kernel (uma vez, antes de tudo)

**Achado real rodando este notebook**: o `%pip install` abaixo muda a versão do `numpy`
instalado, mas o kernel do Colab já tinha uma versão DIFERENTE carregada em memória desde
antes — isso quebra qualquer import de `transformers`/`torch` depois, com
`ValueError: numpy.dtype size changed, may indicate binary incompatibility`. O jeito correto
de evitar isso no Colab é reiniciar o kernel logo depois de instalar, ANTES de importar
qualquer coisa — a célula abaixo faz isso automaticamente.

**A célula vai reiniciar a sessão sozinha — isso é esperado, não um erro.** O Colab vai
mostrar algo como "sessão encerrada inesperadamente"; é só o `os.kill` reiniciando o kernel de
propósito. Depois que reiniciar, **continue rodando as células a partir da próxima** (não
precisa rodar esta de novo).

In [ ]:
# torchao>=0.16.0 (D-torchao, requirements-train.txt): peft==0.19.1 checa a versao
# do torchao na importacao e falha se for menor que 0.16.0 -- o Colab vem com uma
# versao mais antiga por padrao (confirmado: "Found version 0.10.0").
%pip install -q transformers==5.10.2 peft==0.19.1 accelerate==1.10.1 "torchao>=0.16.0"

import os

print("Dependências instaladas — reiniciando o kernel pra evitar incompatibilidade de ABI do numpy...")
os.kill(os.getpid(), 9)

## 2. Montar o Drive e mesclar o adapter no modelo base (GPU, custo alto)

In [ ]:
ADAPTER_DIR = "/content/drive/MyDrive/outputs-logos-1-v2/adapter"  # ajuste se o caminho no seu Drive for diferente
BASE_MODEL = "ibm-granite/granite-4.1-8b"  # bate com configs/train_l4.yaml

from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

assert Path(ADAPTER_DIR).is_dir(), f"pasta não encontrada: {ADAPTER_DIR} — ajuste ADAPTER_DIR acima

In [ ]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# bfloat16 (não 4-bit) — merge precisa de precisão cheia pra aplicar o delta LoRA
# corretamente; a quantização real acontece depois, na conversão pra GGUF (Seção 3).
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
merge_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged = merge_model.merge_and_unload()

MERGED_DIR = "/content/logos3_merged"
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Modelo mesclado salvo em {MERGED_DIR}")

In [ ]:
# Libera a VRAM antes das células de conversão (que não precisam mais de GPU).
del merged, merge_model, base_model
import gc

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Converter pra GGUF e quantizar pra Q4_K_M (CPU, custo baixo)

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
%pip install -q -r /content/llama.cpp/requirements.txt

In [ ]:
# HF -> GGUF f16 (passo intermediário, não é o arquivo final).
GGUF_F16 = "/content/logos3.f16.gguf"
!python /content/llama.cpp/convert_hf_to_gguf.py "{MERGED_DIR}" --outfile "{GGUF_F16}" --outtype f16

import os
print(f"GGUF f16: {os.path.getsize(GGUF_F16) / 1e9:.2f} GB")

**Se a célula acima falhar com "Model architecture not supported"**: tente atualizar o clone
do `llama.cpp` (remova `--depth 1` na célula anterior e rode de novo) — arquiteturas novas
como Granite 4.x demoram a ganhar suporte, mas como você já tem `granite4.1:8b` rodando no
seu Ollama, o suporte existe em algum commit recente.

In [ ]:
# Builda só o binário llama-quantize (mais rápido que o projeto inteiro).
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --config Release -j --target llama-quantize

In [ ]:
GGUF_Q4 = "/content/logos3.Q4_K_M.gguf"
!/content/llama.cpp/build/bin/llama-quantize "{GGUF_F16}" "{GGUF_Q4}" Q4_K_M

import os
print(f"GGUF quantizado (Q4_K_M): {os.path.getsize(GGUF_Q4) / 1e9:.2f} GB")

## 4. Baixar

In [ ]:
import shutil

drive_gguf = "/content/drive/MyDrive/logos3.Q4_K_M.gguf"
shutil.copy(GGUF_Q4, drive_gguf)
print(f"Copiado pro Drive: {drive_gguf} — baixe pela interface web se o download direto abaixo falhar.")

from google.colab import files

files.download(GGUF_Q4)

## 5. No seu PC

```powershell
"FROM ./logos3.Q4_K_M.gguf" | Out-File -Encoding utf8 Modelfile
ollama create logos3 -f Modelfile
ollama list

python scripts/chat_cli.py --ollama-model logos3
```

Modelfile intencionalmente só com `FROM` — o harness manda o prompt em modo `raw`
(`src/inference/ollama_runner.py`), então `TEMPLATE`/`SYSTEM` do Ollama nunca são usados.

Não use `ollama run logos3` direto pra testar o adapter: sem o `chat_cli.py`, nenhum
`<tool_call>` é executado de verdade — o modelo pode gerar um `<tool_result>` inventado sem
nada ter rodado.